In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anthonytherrien/dog-vs-cat")

print("Path to dataset files:", path)

In [8]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms
import numpy as np
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
import numpy as np
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

In [9]:


class ClassificationDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        :param root_dir: Путь к директории с данными. Внутри должно быть N папок, каждая соответствует своему классу.
        :param transform: Трансформации, которые будут применяться к изображениям.
        """
        self.root_dir = root_dir
        self.transform = transform

        # Список всех папок (классов) в корневой директории
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])

        # Количество классов
        self.num_classes = len(self.classes)

        # Создание маппинга класс -> индекс
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

        # Список всех изображений и их классов
        self.samples = []
        for class_name in self.classes:
            class_path = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)
                if os.path.isfile(img_path):
                    self.samples.append((img_path, self.class_to_idx[class_name]))

        # One-hot encoding для классов
        self.one_hot_classes = np.eye(self.num_classes)

    def __len__(self):
        """Возвращает длину датасета"""
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Возвращает элемент датасета по индексу.
        :param idx: Индекс элемента
        :return: Кортеж (изображение, one-hot класс)
        """
        img_path, class_idx = self.samples[idx]

        # Загрузка изображения
        image = Image.open(img_path).convert('RGB')

        # Применение трансформаций (если есть)
        if self.transform:
            image = self.transform(image)

        # One-hot encoding для класса
        label = self.one_hot_classes[class_idx]

        return image, torch.tensor(label, dtype=torch.float32)

In [10]:
dataset = ClassificationDataset(
    root_dir="animals",  
    transform=transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
)

print(f"Количество классов: {dataset.num_classes}")
print(f"Классы: {dataset.classes}")
print(f"Количество образцов: {len(dataset)}")

# Получение первого элемента
image, label = dataset[0]
print(f"Размер изображения: {image.size()}")
print(f"One-hot метка: {label}")

Количество классов: 2
Классы: ['cat', 'dog']
Количество образцов: 1000
Размер изображения: torch.Size([3, 224, 224])
One-hot метка: tensor([1., 0.])


In [11]:

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Количество классов: {dataset.num_classes}")
print(f"Классы: {dataset.classes}")
print(f"Количество образцов: {len(dataset)}")
print(f"Train size: {train_size}, Test size: {test_size}")

# Создание предобученной модели resnet18
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, dataset.num_classes)  # Замена последнего слоя под количество классов
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Процесс обучения
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    print(f"Epoch {epoch+1}/{num_epochs}")
    train_progress = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_progress:
        inputs, labels = inputs.to(device), labels.to(device)
        labels = torch.argmax(labels, dim=1)  # Преобразование one-hot в индексы

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_loss / train_total
    train_acc = train_correct / train_total

    # Тестирование
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            labels = torch.argmax(labels, dim=1)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    test_loss /= test_total
    test_acc = test_correct / test_total

    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Количество классов: 2
Классы: ['cat', 'dog']
Количество образцов: 1000
Train size: 800, Test size: 200


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/10


Epoch 1/10 - Train Loss: 0.1572, Train Acc: 0.9363, Test Loss: 3.2318, Test Acc: 0.6350
Epoch 2/10


Epoch 2/10 - Train Loss: 0.0365, Train Acc: 0.9850, Test Loss: 0.7470, Test Acc: 0.8750
Epoch 3/10


...

Epoch 6/10 - Train Loss: 0.0014, Train Acc: 1.0000, Test Loss: 0.0069, Test Acc: 0.9950
Epoch 7/10


Epoch 7/10 - Train Loss: 0.0003, Train Acc: 1.0000, Test Loss: 0.0059, Test Acc: 0.9950
Epoch 8/10


Epoch 8/10 - Train Loss: 0.0005, Train Acc: 1.0000, Test Loss: 0.0253, Test Acc: 0.9900
Epoch 9/10


Epoch 9/10 - Train Loss: 0.0002, Train Acc: 1.0000, Test Loss: 0.0126, Test Acc: 0.9950
Epoch 10/10


Epoch 10/10 - Train Loss: 0.0001, Train Acc: 1.0000, Test Loss: 0.0118, Test Acc: 0.9950
